# 05A - Sequence Memory Theory

**Guiding question:** when the opening note matters twenty steps later, what path can carry the evidence and the teaching signal?

> **What you already know:** tensors have explicit batch and feature axes, and autograd follows a computation graph.
> **The gap this chapter closes:** a fixed-size network has no reusable state for ordered evidence.
> **What you'll have by the end:** a measured vanilla-memory failure, a visible LSTM cell-state route, and a controlled RNN-versus-LSTM comparison.
> **What this chapter is not:** a framework API tour; implementation contracts belong to 05B.

We keep Melodyne Labs and the familiar word `twinkle`. The opening cue must still matter when the phrase repeats.

| Part | Complaint | Evidence |
|---|---|---|
| 1 | A flat input cannot express order | `(batch, time, feature)` trace |
| 2 | Reused state can still lose old teaching signals | static gradient curve plus animation |
| 3 | Clipping cannot restore forgotten evidence | controlled retention measurement |
| 4 | Memory needs a protected route | LSTM gate storyboard plus animation |
| 5 | A better route still needs real batch contracts | handoff to 05B |


In [ ]:
# -- Setup: deterministic evidence, no training framework required ----------------
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

np.random.seed(42)
torch.manual_seed(42)
plt.rcParams.update({"figure.facecolor": "#1a1a2e", "axes.facecolor": "#1a1a2e"})

melody = "twinkle twinkle"
print(f"Running example: {melody!r}")
print("Question: can the second 'twinkle' still receive evidence from the first?")


## 1 - Order creates a new axis

```mermaid
flowchart LR
    A["Flat features"] --> B["One prediction"]
    C["Ordered timesteps"] --> D["Shared recurrent cell"]
    D --> E["State carried forward"]
    style A fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** if two examples contain the same notes in different orders, can a bag of note counts distinguish them?


In [ ]:
# -- Shape contract: preserve order instead of collapsing it ----------------------
vocab = sorted(set(melody))
token_to_id = {token: index for index, token in enumerate(vocab)}
ids = torch.tensor([[token_to_id[token] for token in melody]], dtype=torch.long)
one_hot = torch.nn.functional.one_hot(ids, num_classes=len(vocab)).float()

forward_counts = one_hot.sum(dim=1)
reversed_counts = one_hot.flip(dims=(1,)).sum(dim=1)
same_bag = torch.equal(forward_counts, reversed_counts)

print(f"Token IDs shape: {tuple(ids.shape)} = (batch, time)")
print(f"One-hot shape:   {tuple(one_hot.shape)} = (batch, time, feature)")
print(f"Forward and reversed bags equal: {same_bag}")
print("Takeaway: counts preserve membership; the time axis preserves order.")
assert same_bag


### The smallest useful notecard

A vanilla recurrent cell reuses one update at every position:

`new state = tanh(current input contribution + previous state contribution)`

The state is a notecard, not a transcript. It must compress the past into a fixed width.


In [ ]:
# -- Manual recurrence: one cell reused through time ------------------------------
x_steps = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
W_x = torch.tensor([[0.7, -0.2], [0.3, 0.8]])
W_h = torch.tensor([[0.6, 0.1], [-0.1, 0.5]])
state = torch.zeros(2)
states = []

for step, x_t in enumerate(x_steps):
    state = torch.tanh(W_x @ x_t + W_h @ state)
    states.append(state.clone())
    print(f"step {step}: state = {[round(value, 3) for value in state.tolist()]}")

print("Takeaway: the parameters stay fixed; only the carried state changes.")


## 2 - The notecard can forget before teaching arrives

```mermaid
flowchart RL
    L["Final loss"] --> H3["state t=3"]
    H3 --> H2["state t=2"]
    H2 --> H1["state t=1"]
    H1 --> X0["opening cue"]
    style L fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H3 fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H2 fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H1 fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style X0 fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

Backpropagation through time follows that chain in reverse. Every transition multiplies the signal again.

**Predict:** with 65% retention per step, which survives more than 1% of the original signal: a cue 5, 10, or 20 steps away?


In [ ]:
# -- Measured vanishing gradient: distance is the only changed variable ----------
distances = np.arange(1, 31)
vanilla_retention = 0.65
vanilla_gradient = vanilla_retention ** distances

for distance in (5, 10, 20):
    print(f"distance {distance:2d}: retained gradient = {vanilla_gradient[distance - 1]:.6f}")

first_below_one_percent = int(distances[np.flatnonzero(vanilla_gradient < 0.01)[0]])
print(f"First distance below 1%: {first_below_one_percent} steps")
print("Takeaway: repeated multiplication makes the opening cue almost unteachable.")


In [ ]:
# -- Static evidence plus animation: the same recorded decay ---------------------
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(distances, vanilla_gradient, color="#f97316", linewidth=3)
ax.axhline(0.01, color="#ef4444", linestyle="--", label="1% retained")
ax.set(yscale="log", xlabel="Distance from the loss", ylabel="Gradient reaching the cue")
ax.set_title("Vanilla route: teaching signal decays with distance", color="white")
ax.tick_params(colors="white")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(9, 4))
ax.set(xlim=(1, 30), ylim=(1e-6, 1), yscale="log")
ax.set_xlabel("Distance from the loss")
ax.set_ylabel("Gradient reaching the cue")
line, = ax.plot([], [], color="#f97316", linewidth=3)
marker, = ax.plot([], [], "o", color="#ef4444")

def update_decay(frame):
    visible = frame + 1
    line.set_data(distances[:visible], vanilla_gradient[:visible])
    marker.set_data([distances[frame]], [vanilla_gradient[frame]])
    ax.set_title(
        f"Distance {distances[frame]}: {vanilla_gradient[frame]:.6f} retained",
        color="white",
    )
    return line, marker

animation = FuncAnimation(fig, update_decay, frames=len(distances), interval=90, blit=True)
plt.close(fig)
print("Watch the marker: distance grows while the usable teaching signal collapses.")
display(HTML(animation.to_jshtml()))


### That failed, but clipping cannot restore it

Clipping caps a gradient after it exists. It can tame an exploding update; it cannot turn `0.00018` back into useful evidence.

**Your turn:** change the retention factor and predict the first distance below 1%.


In [ ]:
# -- Your turn: change one memory-route variable ---------------------------------
# CHANGE THIS: try 0.50 or 0.85
retention_to_test = 0.75
tested_curve = retention_to_test ** distances
below = np.flatnonzero(tested_curve < 0.01)
crossing = int(distances[below[0]]) if len(below) else None
print(f"Retention {retention_to_test:.2f}: first distance below 1% = {crossing}")
print("Check: a larger retention factor must push the crossing farther away.")


## 3 - LSTM creates a controlled memory route

```mermaid
flowchart LR
    C0["old cell state"] --> F["forget gate"]
    X["current input"] --> I["input gate"]
    X --> G["candidate"]
    F --> C1["new cell state"]
    I --> C1
    G --> C1
    C1 --> O["output gate"]
    O --> H["visible hidden state"]
    style C0 fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style X fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style I fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C1 fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style O fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** when the forget gate stays near 1 and the input gate closes, should the cell state drift, reset, or remain nearly flat?


In [ ]:
# -- Gate evidence: record actual cell-state updates ------------------------------
forget = np.array([0.95, 0.95, 0.90, 0.98, 0.97, 0.96])
input_gate = np.array([0.80, 0.10, 0.05, 0.05, 0.05, 0.05])
candidate = np.array([0.90, -0.20, 0.10, 0.00, 0.05, 0.00])
output = np.array([0.70, 0.75, 0.80, 0.80, 0.85, 0.85])
cell = 0.0
cell_history = []
hidden_history = []

for step in range(len(forget)):
    cell = forget[step] * cell + input_gate[step] * candidate[step]
    hidden = output[step] * np.tanh(cell)
    cell_history.append(cell)
    hidden_history.append(hidden)

for step, (cell_value, hidden_value) in enumerate(zip(cell_history, hidden_history)):
    print(f"step {step}: cell={cell_value:.3f}, visible hidden={hidden_value:.3f}")
print("Takeaway: the cell can carry evidence even when little new content is written.")


In [ ]:
# -- Static storyboard and animation: gates control one carried state ------------
steps = np.arange(len(cell_history))
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
axes[0].plot(steps, forget, label="forget", color="#60a5fa", marker="o")
axes[0].plot(steps, input_gate, label="input", color="#f59e0b", marker="o")
axes[0].plot(steps, output, label="output", color="#34d399", marker="o")
axes[0].set_ylabel("Gate value")
axes[0].legend(ncol=3)
axes[1].plot(steps, cell_history, label="cell state", color="#a78bfa", marker="o")
axes[1].plot(steps, hidden_history, label="visible hidden", color="#f472b6", marker="o")
axes[1].set(xlabel="Timestep", ylabel="State value")
axes[1].legend()
plt.show()

fig, ax = plt.subplots(figsize=(9, 4))
ax.set(xlim=(-0.3, len(steps) - 0.7), ylim=(-0.1, 0.9))
ax.set_xlabel("Timestep")
ax.set_ylabel("Recorded state")
cell_line, = ax.plot([], [], color="#a78bfa", linewidth=3, marker="o", label="cell")
hidden_line, = ax.plot([], [], color="#f472b6", linewidth=2, marker="o", label="hidden")
ax.legend()

def update_cell(frame):
    visible = frame + 1
    cell_line.set_data(steps[:visible], cell_history[:visible])
    hidden_line.set_data(steps[:visible], hidden_history[:visible])
    ax.set_title(
        f"f={forget[frame]:.2f}, i={input_gate[frame]:.2f}, o={output[frame]:.2f}",
        color="white",
    )
    return cell_line, hidden_line

gate_animation = FuncAnimation(fig, update_cell, frames=len(steps), interval=550, blit=True)
plt.close(fig)
print("Watch the cell line stay stable while the gates change what is written and exposed.")
display(HTML(gate_animation.to_jshtml()))


## 4 - Controlled RNN versus LSTM memory routes

Change one thing: the repeated retention multiplier. The delay, loss location, and initial signal stay fixed.

```mermaid
flowchart LR
    S["Opening cue"] --> R["Vanilla route: x 0.65 each step"]
    S --> L["LSTM cell route: x 0.95 each step"]
    R --> RF["Weak teaching signal"]
    L --> LF["Usable teaching signal"]
    style S fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style L fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style RF fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style LF fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Controlled comparison: same delay, different memory route -------------------
lstm_gradient = 0.95 ** distances
delay = 20
vanilla_at_delay = vanilla_gradient[delay - 1]
lstm_at_delay = lstm_gradient[delay - 1]
ratio = lstm_at_delay / vanilla_at_delay

fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogy(distances, vanilla_gradient, label="vanilla route (0.65)", linewidth=3)
ax.semilogy(distances, lstm_gradient, label="LSTM cell route (0.95)", linewidth=3)
ax.set(xlabel="Distance from loss", ylabel="Gradient reaching cue")
ax.legend()
plt.show()

print(f"At distance {delay}: vanilla={vanilla_at_delay:.6f}, LSTM route={lstm_at_delay:.6f}")
print(f"Controlled retention advantage: {ratio:,.0f}x")
print("Boundary: this isolates the route; it does not claim every trained LSTM always wins.")
assert lstm_at_delay > vanilla_at_delay


## 5 - Evidence scorecard and handoff

| Opening question | Evidence | Answer |
|---|---|---|
| Why a time axis? | forward and reversed bags were identical | order needs `(batch, time, feature)` |
| Why can a vanilla RNN miss old cues? | measured repeated gradient decay | the teaching route crosses every transition |
| What do LSTM gates buy? | recorded gate and cell trajectories | controlled writing, retaining, and exposing |
| Is the comparison universal? | one-variable route experiment | no; it isolates mechanism, not all training conditions |

**Built and measured:** recurrence, gradient decay, LSTM state updates, controlled route comparison.

**Explained or illustrated:** BPTT, exploding gradients, clipping boundary.

**Named and out of scope:** packed batches, padding-aware loss, and causal tests. Those are the job of [05B](05B-sequence-memory-pytorch-lab.ipynb).

The unresolved question is now practical: how do real variable-length batches preserve the last real token, ignore padding, and keep future inputs from leaking backward?
